# Geography and Mapping

This notebook aims to explore the geographical aspects of SPD's calls data, breaking down call volume, proportion of events, and types of events in each neighborhood. In the cells below we go over methods of visualizing the data, findings about how certain types crimes are clustered together, and how specific areas tend to have a higher level of activity. 

## Data Loading and EDA

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import re
from dashboard.spd_snapshot import load_spd_call_snapshot
from dashboard.spd_eda import summarize_spd_calls
from dashboard.spd_event_volume import load_spd_event_volume
import html
import folium
from folium.plugins import MarkerCluster
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
from plotly.subplots import make_subplots
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')


df, metadata = load_spd_call_snapshot(
    PROJECT_ROOT / "data" / "processed"
)

display(df.head())
summary = summarize_spd_calls(df)
print(summary)

EVENT_ID_COLUMN = "cad_event_number"
ROW_ID_COLUMN = "call_sign_dispatch_id"
TIME_COLUMN = "cad_event_original_time_queued"

LAT_COL = "dispatch_latitude"
LON_COL = "dispatch_longitude"

PLOTLY_TEMPLATE = "plotly_dark"
PLOT_BG = "#545455"
PAPER_BG = "#111111"

# Rough Seattle bounding box.
# This is a quick sanity check, not a true city boundary.
SEATTLE_LAT_MIN = 47.45
SEATTLE_LAT_MAX = 47.75
SEATTLE_LON_MIN = -122.46
SEATTLE_LON_MAX = -122.20

In [ ]:
geo_df = df.copy()

geo_df[TIME_COLUMN] = pd.to_datetime(
    geo_df[TIME_COLUMN],
    errors="coerce"
)

geo_df[LAT_COL] = pd.to_numeric(
    geo_df[LAT_COL],
    errors="coerce"
)

geo_df[LON_COL] = pd.to_numeric(
    geo_df[LON_COL],
    errors="coerce"
)

text_cols = [
    "event_group",
    "dispatch_neighborhood",
    "dispatch_precinct",
    "dispatch_sector",
    "dispatch_beat",
]

for col in text_cols:
    if col in geo_df.columns:
        geo_df[col] = (
            geo_df[col]
            .astype("string")
            .str.strip()
            .str.lower()
        )

Below we can see some important metrics about the amounts and proportions of entries with mappable coordinates. As briefly explored in the `Schema and Data Quality` notebook about 75.74% of the unique call events have mappable coordinates (brief note: in the notebook it was reported as 24.26% missing but here we are specifically looking at the rest that are mappable here). We can also see that while we have coordinates for the majority of events in the data, there are some (2.0% of the events to be exact) that have coordinates that are clearly placeholders ([-1, -1] is a location that is nowhere near Seattle, in fact it is in the Atlantic Ocean)

In [ ]:
# Coordinate validity checks
geo_df["is_placeholder_coord"] = (
    ((geo_df[LAT_COL] == -1) & (geo_df[LON_COL] == -1))
    | ((geo_df[LAT_COL] == 0) & (geo_df[LON_COL] == 0))
)

geo_df["has_valid_coords"] = (
    geo_df[LAT_COL].notna()
    & geo_df[LON_COL].notna()
    & geo_df[LAT_COL].between(-90, 90)
    & geo_df[LON_COL].between(-180, 180)
    & ~geo_df["is_placeholder_coord"]
)

geo_df["inside_seattle_bbox"] = (
    geo_df["has_valid_coords"]
    & geo_df[LAT_COL].between(SEATTLE_LAT_MIN, SEATTLE_LAT_MAX)
    & geo_df[LON_COL].between(SEATTLE_LON_MIN, SEATTLE_LON_MAX)
)

total_rows = len(geo_df)
total_events = geo_df[EVENT_ID_COLUMN].nunique()

coordinate_summary = pd.DataFrame({
    "Metric": [
        "Total dispatch rows",
        "Rows with placeholder coordinates",
        "Rows with valid coordinates",
        "Rows inside rough Seattle bounding box",
        "Unique CAD events",
        "Unique CAD events with placeholder coordinates",
        "Unique CAD events with valid coordinates",
        "Unique CAD events inside rough Seattle bounding box",
    ],
    "Value": [
        total_rows,
        geo_df["is_placeholder_coord"].sum(),
        geo_df["has_valid_coords"].sum(),
        geo_df["inside_seattle_bbox"].sum(),
        total_events,
        geo_df.loc[geo_df["is_placeholder_coord"], EVENT_ID_COLUMN].nunique(),
        geo_df.loc[geo_df["has_valid_coords"], EVENT_ID_COLUMN].nunique(),
        geo_df.loc[geo_df["inside_seattle_bbox"], EVENT_ID_COLUMN].nunique(),
    ],
})

coordinate_summary["Percentage"] = np.where(
    coordinate_summary["Metric"].str.contains("Rows|dispatch"),
    round((coordinate_summary["Value"] / total_rows * 100), ndigits=2),
    round((coordinate_summary["Value"] / total_events * 100), ndigits=2),
)

# Formatting for readability (nightmare)
coordinate_summary['Percentage'] = coordinate_summary['Percentage'].map(
    lambda x: f"{float(x):.1f}%" if pd.notnull(x) and str(x).strip() != "" else ""
)
coordinate_summary['Value'] = coordinate_summary['Value'].apply(lambda x: f"{x:,.1f}" if pd.notnull(x) else "")

coordinate_summary_print = coordinate_summary.set_index('Metric').T
coordinate_summary_print.index.name = None
coordinate_summary_print = coordinate_summary_print.style.set_properties(**{
    'text-align': 'center'
}).set_table_styles([{
    'selector': 'th',
    'props': [('text-align', 'center')]
}])
coordinate_summary_print


In [ ]:
mappable_events = (
    geo_df[
        geo_df["inside_seattle_bbox"]
        & geo_df[EVENT_ID_COLUMN].notna()
    ]
    .sort_values(TIME_COLUMN, ascending=False)
    .drop_duplicates(subset=EVENT_ID_COLUMN)
    .copy()
)

Below we see a breakdown of the top 20 dispatch neighborhoods by volume of events, in which we can see that Capitol Hill is the neighborhood with the highest volume of events followed by Downtown, Northgate, International District, and South Lake Union/Cascade.

In [ ]:
# One row per CAD event for geography summaries
# This prevents dispatch/unit rows from inflating call counts.

event_geo = (
    geo_df
    .dropna(subset=[EVENT_ID_COLUMN])
    .sort_values(TIME_COLUMN)
    .groupby(EVENT_ID_COLUMN, as_index=False)
    .agg(
        event_time=(TIME_COLUMN, "min"),
        event_group=("event_group", "first"),
        dispatch_neighborhood=("dispatch_neighborhood", "first"),
        dispatch_precinct=("dispatch_precinct", "first"),
        dispatch_sector=("dispatch_sector", "first"),
        dispatch_beat=("dispatch_beat", "first"),
        dispatch_latitude=(LAT_COL, "first"),
        dispatch_longitude=(LON_COL, "first"),
        has_valid_coords=("has_valid_coords", "max"),
        inside_seattle_bbox=("inside_seattle_bbox", "max"),
        is_placeholder_coord=("is_placeholder_coord", "max"),
        dispatch_records=(ROW_ID_COLUMN, "nunique"),
    )
)

event_geo.head()

neighborhood_counts = (
    event_geo
    .dropna(subset=["dispatch_neighborhood"])
    .query("dispatch_neighborhood != '-'")
    .groupby("dispatch_neighborhood", as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        dispatch_records=("dispatch_records", "sum"),
        events_with_valid_coords=("has_valid_coords", "sum"),
        events_inside_seattle_bbox=("inside_seattle_bbox", "sum"),
    )
    .sort_values("unique_call_events", ascending=False)
    .reset_index(drop=True)
)

neighborhood_counts["pct_with_valid_coords"] = (
    neighborhood_counts["events_with_valid_coords"]
    / neighborhood_counts["unique_call_events"]
    * 100
)

neighborhood_counts.head(20)

fig = px.bar(
    neighborhood_counts.head(20),
    x="unique_call_events",
    y="dispatch_neighborhood",
    orientation="h",
    title="Top 20 Dispatch Neighborhoods by Unique CAD Events",
    labels={
        "unique_call_events": "Unique CAD Events",
        "dispatch_neighborhood": "Dispatch Neighborhood",
    },
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="Unique CAD Events",
    yaxis_title="Dispatch Neighborhood",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

fig.show()

Below we see the volume of calls (within the mappable events data) in each dispatch precinct. The North and West precincts have the highest call volume with both having over 90,000 events within the last year.

In [ ]:
precinct_counts = (
    event_geo
    .dropna(subset=["dispatch_precinct"])
    .query("dispatch_precinct != '-'")
    .groupby("dispatch_precinct", as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        dispatch_records=("dispatch_records", "sum"),
        events_with_valid_coords=("has_valid_coords", "sum"),
        events_inside_seattle_bbox=("inside_seattle_bbox", "sum"),
    )
    .sort_values("unique_call_events", ascending=False)
    .reset_index(drop=True)
)

precinct_counts["pct_with_valid_coords"] = (
    precinct_counts["events_with_valid_coords"]
    / precinct_counts["unique_call_events"]
    * 100
)

precinct_counts

fig = px.bar(
    precinct_counts,
    x="dispatch_precinct",
    y="unique_call_events",
    title="Unique CAD Events by Dispatch Precinct",
    labels={
        "dispatch_precinct": "Dispatch Precinct",
        "unique_call_events": "Unique CAD Events",
    },
    text="unique_call_events",
)

fig.update_traces(
    texttemplate="%{text:,}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    xaxis_title="Dispatch Precinct",
    yaxis_title="Unique CAD Events",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

fig.show()

In the chart below we can see the top 10 event groups within the highest volume dispatch neighborhoods. Calls that settle into the event groups of "assist other agency", "assist public", "traffic", et cetera have much higher volume relative to crimes that most people have greater concern about. Aside from "theft", "prowler", and perhaps "crisis complaint" there isn't much within the figure that reveals information about crimes committed that affect other people. 

In [ ]:
top_neighborhoods = neighborhood_counts.head(12)["dispatch_neighborhood"].tolist()

top_event_groups = (
    event_geo
    .dropna(subset=["event_group"])
    .query("event_group != '-'")
    .groupby("event_group")[EVENT_ID_COLUMN]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .index
    .tolist()
)

neighborhood_event_group = (
    event_geo
    .dropna(subset=["dispatch_neighborhood", "event_group"])
    .query("dispatch_neighborhood in @top_neighborhoods")
    .query("event_group in @top_event_groups")
    .groupby(["dispatch_neighborhood", "event_group"], as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique")
    )
)

neighborhood_event_group.head()

fig = px.bar(
    neighborhood_event_group,
    x="unique_call_events",
    y="dispatch_neighborhood",
    color="event_group",
    orientation="h",
    title="Top Event Groups within High-Volume Dispatch Neighborhoods",
    labels={
        "unique_call_events": "Unique CAD Events",
        "dispatch_neighborhood": "Dispatch Neighborhood",
        "event_group": "Event Group",
    },
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="Unique CAD Events",
    yaxis_title="Dispatch Neighborhood",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

fig.show()

Below we see a figure much like the one above, we have stacked bars showing the volume of calls for different event groups within each neighborhood but this time we have limited the event groups to those relating to crimes/violent crimes. The figure shows the data for the top 12 neighborhoods by call volume, and in each neighborhood we can see that a large proportion of the calls are `assaults`, `domestic disturbance/violence`, and `theft`. Once we take a closer look at the top 5 neighborhoods by call volume (Capitol Hill, Downtown Commercial, SLU/Cascade, Northgate, and Chinatown/International District) we can see that there is a greater prevalence of narcotics calls within these specific groups. Chinatown, Downtown, and Capitol Hill have ~1000 calls relating to narcotics (note that this is within the 2025-2026 data *with mappable coordinates*). The top 5 neighborhoods also have a higher prevalence of sex offences (non-rape) compared to the rest, Capitol Hill in particular has nearly 200 of these calls while the rest in the top 5 have about half as many. 

In [ ]:
# Event groups by neighborhood, excluding less useful / overly broad categories
top_neighborhoods = neighborhood_counts.head(12)["dispatch_neighborhood"].tolist()

exclude_event_groups = [
    "traffic", "assist public", "alarms",
    "miscellaneous", 'automobiles', 'disturbance',
    'warrant services/order', 'traffic', 'suspicious circumstances',
    'premise checks', 'assist other agency', 'assist public',
    'other', 'alarms, false', 'directed patrol',
    'assigned duty', 'property', 'follow-ups',
    'administrative', 'fraud & forgery', 'unkown-ani/ali',
    'info & radio broadcast', 'public gatherings', 'harbor (water)',
    'assist the officer', 'animal complaint', 'bias',
    'custodial interference', 'swatting', 'detox',
]

neighborhood_event_group = (
    event_geo
    .dropna(subset=["dispatch_neighborhood", "event_group"])
    .query("dispatch_neighborhood != '-'")
    .query("event_group != '-'")
    .query("event_group not in @exclude_event_groups")
    .query("dispatch_neighborhood in @top_neighborhoods")
    .groupby(["dispatch_neighborhood", "event_group"], as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique")
    )
)

# Order neighborhoods by total remaining call volume
neighborhood_order = (
    neighborhood_event_group
    .groupby("dispatch_neighborhood")["unique_call_events"]
    .sum()
    .sort_values(ascending=True)
    .index
    .tolist()
)

neighborhood_event_group.head()

fig = px.bar(
    neighborhood_event_group,
    x="unique_call_events",
    y="dispatch_neighborhood",
    color="event_group",
    orientation="h",
    title="Unique CAD Events by Dispatch Neighborhood and Event Group",
    labels={
        "unique_call_events": "Unique CAD Events",
        "dispatch_neighborhood": "Dispatch Neighborhood",
        "event_group": "Event Group",
    },
    category_orders={
        "dispatch_neighborhood": neighborhood_order,
    },
)

fig.update_layout(
    barmode="stack",
    xaxis_title="Unique CAD Events",
    yaxis_title="Dispatch Neighborhood",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
    height=max(700, len(neighborhood_order) * 22),
)

visible_event_groups = [
    "assaults",
    "burglary",
    "domestic disturbance/violence",
    "kidnap",
    "rape",
    "robbery",
    "sex offenses (non-rape)",
    "theft",
    "narcotics",
    "homicide",
]

for trace in fig.data:
    if trace.name not in visible_event_groups:
        trace.visible = "legendonly"


fig.show()

Below we can see the same figure as above but for the *bottom* 5 neighborhoods by call volume. One of the notable differences is the larger proportion of calls in each neighborhood that are `crisis complaints` or `domestic disturbance/violence`

In [ ]:
# Event groups by neighborhood, excluding less useful / overly broad categories
bottom_neighborhoods = neighborhood_counts.tail(12)["dispatch_neighborhood"].tolist()

exclude_event_groups = [
    "traffic", "assist public", "alarms",
    "miscellaneous", 'automobiles', 'disturbance',
    'warrant services/order', 'traffic', 'suspicious circumstances',
    'premise checks', 'assist other agency', 'assist public',
    'other', 'alarms, false', 'directed patrol',
    'assigned duty', 'property', 'follow-ups',
    'administrative', 'fraud & forgery', 'unkown-ani/ali',
    'info & radio broadcast', 'public gatherings', 'harbor (water)',
    'assist the officer', 'animal complaint', 'bias',
    'custodial interference', 'swatting', 'detox',
]

neighborhood_event_group = (
    event_geo
    .dropna(subset=["dispatch_neighborhood", "event_group"])
    .query("dispatch_neighborhood != '-'")
    .query("event_group != '-'")
    .query("event_group not in @exclude_event_groups")
    .query("dispatch_neighborhood in @bottom_neighborhoods")
    .groupby(["dispatch_neighborhood", "event_group"], as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique")
    )
)

# Order neighborhoods by total remaining call volume
neighborhood_order = (
    neighborhood_event_group
    .groupby("dispatch_neighborhood")["unique_call_events"]
    .sum()
    .sort_values(ascending=True)
    .index
    .tolist()
)

neighborhood_event_group.head()

fig = px.bar(
    neighborhood_event_group,
    x="unique_call_events",
    y="dispatch_neighborhood",
    color="event_group",
    orientation="h",
    title="Unique CAD Events by Dispatch Neighborhood and Event Group",
    labels={
        "unique_call_events": "Unique CAD Events",
        "dispatch_neighborhood": "Dispatch Neighborhood",
        "event_group": "Event Group",
    },
    category_orders={
        "dispatch_neighborhood": neighborhood_order,
    },
)

fig.update_layout(
    barmode="stack",
    xaxis_title="Unique CAD Events",
    yaxis_title="Dispatch Neighborhood",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
    height=max(700, len(neighborhood_order) * 22),
)

visible_event_groups = [
    "assaults",
    "burglary",
    "domestic disturbance/violence",
    "kidnap",
    "rape",
    "robbery",
    "sex offenses (non-rape)",
    "theft",
    "narcotics",
    "homicide",
]

for trace in fig.data:
    if trace.name not in visible_event_groups:
        trace.visible = "legendonly"


fig.show()

In [ ]:
recent_mappable = (
    mappable_events
    .sort_values(TIME_COLUMN, ascending=False)
    .head(1500)
    .copy()
)

recent_mappable[
    [
        EVENT_ID_COLUMN,
        TIME_COLUMN,
        "event_group",
        "dispatch_neighborhood",
        LAT_COL,
        LON_COL,
    ]
].head()
print('\n')

## Chloropleth Mapping

In [ ]:
# ------------------------------------------------------------
# Official SPD MCPP boundary setup + event spatial join + choropleth
# Replaces the manual neighborhood-atlas crosswalk approach
# ------------------------------------------------------------

from pathlib import Path
import json
import pandas as pd
import geopandas as gpd
import plotly.express as px

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BOUNDARY_DIR = PROJECT_ROOT / "data" / "external" / "boundaries"
BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

GEO_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "geography"
GEO_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

mcpp_geojson_path = BOUNDARY_DIR / "spd_mcpp_boundaries.geojson"
processed_mcpp_geojson_path = GEO_PROCESSED_DIR / "spd_mcpp_boundaries.geojson"

MCPP_GEOJSON_URL = (
    "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/ArcGIS/rest/services/"
    "SPD_Boundaries/FeatureServer/0/query"
    "?where=1%3D1"
    "&outFields=*"
    "&outSR=4326"
    "&f=geojson"
)

# ------------------------------------------------------------
# Load official MCPP polygons
# ------------------------------------------------------------

if mcpp_geojson_path.exists():
    mcpp_boundaries = gpd.read_file(mcpp_geojson_path)
    print(f"Loaded cached MCPP boundaries from: {mcpp_geojson_path}")
else:
    mcpp_boundaries = gpd.read_file(MCPP_GEOJSON_URL)
    print("Downloaded MCPP boundaries from ArcGIS service.")

# Standardize CRS
if mcpp_boundaries.crs is not None:
    mcpp_boundaries = mcpp_boundaries.to_crs(epsg=4326)
else:
    mcpp_boundaries = mcpp_boundaries.set_crs(epsg=4326)

# Clean column names
mcpp_boundaries.columns = [
    col.lower().strip()
    for col in mcpp_boundaries.columns
]

print("MCPP boundary columns:")
print(mcpp_boundaries.columns.tolist())

# ------------------------------------------------------------
# Standardize MCPP name fields
# Works for both raw ArcGIS data and cleaned cached GeoJSON files
# ------------------------------------------------------------

if "mcpp_neighborhood" in mcpp_boundaries.columns:
    mcpp_boundaries["mcpp_neighborhood"] = (
        mcpp_boundaries["mcpp_neighborhood"]
        .astype("string")
        .str.strip()
        .str.lower()
    )
elif "neighborhood" in mcpp_boundaries.columns:
    mcpp_boundaries["mcpp_neighborhood"] = (
        mcpp_boundaries["neighborhood"]
        .astype("string")
        .str.strip()
        .str.lower()
    )
else:
    raise KeyError(
        "Could not find a neighborhood column. "
        "Expected either 'neighborhood' or 'mcpp_neighborhood'. "
        f"Available columns: {mcpp_boundaries.columns.tolist()}"
    )

if "mcpp_precinct" in mcpp_boundaries.columns:
    mcpp_boundaries["mcpp_precinct"] = (
        mcpp_boundaries["mcpp_precinct"]
        .astype("string")
        .str.strip()
        .str.lower()
    )
elif "precinct" in mcpp_boundaries.columns:
    mcpp_boundaries["mcpp_precinct"] = (
        mcpp_boundaries["precinct"]
        .astype("string")
        .str.strip()
        .str.lower()
    )
else:
    mcpp_boundaries["mcpp_precinct"] = pd.NA

# If objectid is missing, create a stable boundary id
if "objectid" not in mcpp_boundaries.columns:
    mcpp_boundaries["objectid"] = range(1, len(mcpp_boundaries) + 1)

# Keep useful columns
mcpp_boundaries = mcpp_boundaries[
    [
        "objectid",
        "mcpp_neighborhood",
        "mcpp_precinct",
        "geometry",
    ]
].copy()

# Make a unique plotting id
mcpp_boundaries["plot_feature_id"] = (
    mcpp_boundaries["objectid"]
    .astype(str)
)

# Save official boundary layer locally
mcpp_boundaries.to_file(mcpp_geojson_path, driver="GeoJSON")
mcpp_boundaries.to_file(processed_mcpp_geojson_path, driver="GeoJSON")

print()
print(f"MCPP boundaries loaded: {len(mcpp_boundaries):,}")
print(f"Saved external copy to: {mcpp_geojson_path}")
print(f"Saved processed copy to: {processed_mcpp_geojson_path}")

display(mcpp_boundaries.head())

# ------------------------------------------------------------
# Create event_mcpp from mappable_events + official MCPP polygons
# ------------------------------------------------------------

mappable_events_for_join = mappable_events.copy()

mappable_events_for_join[LAT_COL] = pd.to_numeric(
    mappable_events_for_join[LAT_COL],
    errors="coerce"
)

mappable_events_for_join[LON_COL] = pd.to_numeric(
    mappable_events_for_join[LON_COL],
    errors="coerce"
)

mappable_events_for_join = mappable_events_for_join[
    mappable_events_for_join[EVENT_ID_COLUMN].notna()
    & mappable_events_for_join[LAT_COL].notna()
    & mappable_events_for_join[LON_COL].notna()
].copy()

# Make sure there is only one row per CAD event
if TIME_COLUMN in mappable_events_for_join.columns:
    mappable_events_for_join[TIME_COLUMN] = pd.to_datetime(
        mappable_events_for_join[TIME_COLUMN],
        errors="coerce"
    )

    mappable_events_for_join = (
        mappable_events_for_join
        .sort_values(TIME_COLUMN, ascending=False)
        .drop_duplicates(subset=EVENT_ID_COLUMN)
        .copy()
    )
else:
    mappable_events_for_join = (
        mappable_events_for_join
        .drop_duplicates(subset=EVENT_ID_COLUMN)
        .copy()
    )

event_points_gdf = gpd.GeoDataFrame(
    mappable_events_for_join,
    geometry=gpd.points_from_xy(
        mappable_events_for_join[LON_COL],
        mappable_events_for_join[LAT_COL],
    ),
    crs="EPSG:4326",
)

# Rename boundary id before spatial join to avoid column-name collisions
mcpp_boundaries_for_join = (
    mcpp_boundaries[
        [
            "objectid",
            "mcpp_neighborhood",
            "mcpp_precinct",
            "geometry",
        ]
    ]
    .rename(columns={"objectid": "mcpp_objectid"})
    .copy()
)

event_mcpp = gpd.sjoin(
    event_points_gdf,
    mcpp_boundaries_for_join,
    how="left",
    predicate="within",
)

event_mcpp = event_mcpp.drop(columns=["index_right"], errors="ignore")

# ------------------------------------------------------------
# QA checks
# ------------------------------------------------------------

total_mappable_events = event_points_gdf[EVENT_ID_COLUMN].nunique()

matched_mcpp_events = event_mcpp.loc[
    event_mcpp["mcpp_neighborhood"].notna(),
    EVENT_ID_COLUMN
].nunique()

unmatched_mcpp_events = total_mappable_events - matched_mcpp_events

print()
print(f"Mappable unique CAD events: {total_mappable_events:,}")
print(f"Events matched to MCPP polygon: {matched_mcpp_events:,}")
print(f"Events not matched to MCPP polygon: {unmatched_mcpp_events:,}")

if total_mappable_events > 0:
    print(f"Match rate: {matched_mcpp_events / total_mappable_events * 100:.2f}%")

display(event_mcpp.head())


MCPP_LOOKUP_DIR = PROJECT_ROOT / "data" / "processed" / "geography"
MCPP_LOOKUP_DIR.mkdir(parents=True, exist_ok=True)

mcpp_event_lookup_path = MCPP_LOOKUP_DIR / "event_mcpp_lookup.parquet"

event_mcpp_lookup = (
    event_mcpp[
        [
            EVENT_ID_COLUMN,
            "mcpp_neighborhood",
            "mcpp_precinct",
        ]
    ]
    .dropna(subset=[EVENT_ID_COLUMN])
    .drop_duplicates(subset=EVENT_ID_COLUMN)
    .copy()
)

event_mcpp_lookup.to_parquet(mcpp_event_lookup_path, index=False)

print(f"Saved MCPP event lookup to: {mcpp_event_lookup_path}")
display(event_mcpp_lookup.head())

# ------------------------------------------------------------
# Official MCPP choropleth: unique CAD events by MCPP neighborhood
# ------------------------------------------------------------

mcpp_event_counts = (
    event_mcpp
    .dropna(subset=["mcpp_neighborhood"])
    .groupby("mcpp_neighborhood", as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        dispatch_records=(ROW_ID_COLUMN, "nunique"),
    )
)

mcpp_choropleth_gdf = mcpp_boundaries.merge(
    mcpp_event_counts,
    on="mcpp_neighborhood",
    how="left",
)

mcpp_choropleth_gdf["unique_call_events"] = (
    mcpp_choropleth_gdf["unique_call_events"]
    .fillna(0)
    .astype(int)
)

mcpp_choropleth_gdf["dispatch_records"] = (
    mcpp_choropleth_gdf["dispatch_records"]
    .fillna(0)
    .astype(int)
)

mcpp_choropleth_gdf["plot_feature_id"] = (
    mcpp_choropleth_gdf["objectid"]
    .astype(str)
)

mcpp_geojson = json.loads(mcpp_choropleth_gdf.to_json())

fig = px.choropleth_mapbox(
    mcpp_choropleth_gdf,
    geojson=mcpp_geojson,
    locations="plot_feature_id",
    featureidkey="properties.plot_feature_id",
    color="unique_call_events",
    hover_name="mcpp_neighborhood",
    hover_data={
        "mcpp_precinct": True,
        "unique_call_events": ":,",
        "dispatch_records": ":,",
        "plot_feature_id": False,
    },
    center={
        "lat": 47.6062,
        "lon": -122.3321,
    },
    zoom=10,
    mapbox_style="carto-darkmatter",
    title="SPD Unique CAD Events by Official MCPP Neighborhood",
    opacity=0.75,
)

fig.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "MCPP precinct: %{customdata[0]}<br>"
        "Unique CAD events: %{customdata[1]:,}<br>"
        "Dispatch records: %{customdata[2]:,}"
        "<extra></extra>"
    )
)

fig.update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    },
)

fig.show()

Below we see the count of events within each neighborhood on a chloropleth map. The shading being dependent on the count of events within each neighborhood gives us an intuitive way of visualizing which neighborhoods have higher volume. While most regions are a dark shade, Capitol Hill, Downtown, and Northgate pop out with a yellow or light orange tint.

In [ ]:
neighborhood_geojson = json.loads(mcpp_choropleth_gdf.to_json())

fig = px.choropleth_mapbox(
    mcpp_choropleth_gdf,
    geojson=neighborhood_geojson,
    locations="plot_feature_id",
    featureidkey="properties.plot_feature_id",
    color="unique_call_events",
    hover_name="mcpp_neighborhood",
    hover_data={
        "mcpp_precinct": True,
        "unique_call_events": ":,",
        "dispatch_records": ":,",
        "plot_feature_id": False,
    },
    center={
        "lat": 47.6062,
        "lon": -122.3321,
    },
    zoom=10,
    mapbox_style="carto-darkmatter",
    title="SPD Unique CAD Events by Official MCPP Neighborhood",
    opacity=0.75,
)

fig.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "MCPP precinct: %{customdata[0]}<br>"
        "Unique CAD events: %{customdata[1]:,}<br>"
        "Dispatch records: %{customdata[2]:,}"
        "<extra></extra>"
    )
)

fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
)

fig.show()

## Filtered Folium maps

In [ ]:
IMPORTANT_EVENT_GROUPS = [
    "assaults",
    "burglary",
    "domestic disturbance/violence",
    "kidnap",
    "rape",
    "robbery",
    "sex offenses (non-rape)",
    "theft",
    "narcotics",
    "homicide",
    "weapon, person with"
]

SEATTLE_CENTER = [47.6062, -122.3321]
PLOTLY_MAP_STYLE = "carto-darkmatter"
print(f"Mappable unique CAD events: {len(mappable_events):,}")
mappable_events.head()

In [ ]:
latest_event_time = mappable_events[TIME_COLUMN].max()
latest_event_date = latest_event_time.normalize()

previous_complete_day = latest_event_date - pd.Timedelta(days=1)

last_week_start = latest_event_time - pd.Timedelta(days=7)
last_month_start = latest_event_time - pd.Timedelta(days=30)

print(f"Latest event time in data: {latest_event_time}")
print(f"Previous complete day: {previous_complete_day.date()}")
print(f"Last week start: {last_week_start}")
print(f"Last month start: {last_month_start}")

previous_day_events = mappable_events[
    mappable_events[TIME_COLUMN].dt.normalize() == previous_complete_day
].copy()

last_week_events = mappable_events[
    mappable_events[TIME_COLUMN] >= last_week_start
].copy()

last_month_events = mappable_events[
    mappable_events[TIME_COLUMN] >= last_month_start
].copy()

window_summary = pd.DataFrame({
    "window": [
        "previous complete day",
        "last 7 days",
        "last 30 days",
    ],
    "unique_call_events": [
        previous_day_events[EVENT_ID_COLUMN].nunique(),
        last_week_events[EVENT_ID_COLUMN].nunique(),
        last_month_events[EVENT_ID_COLUMN].nunique(),
    ],
})

print(window_summary)

In [ ]:
def make_seattle_folium_base_map():
    fmap = folium.Map(
        location=SEATTLE_CENTER,
        zoom_start=11,
        min_zoom=10,
        max_zoom=15,
        tiles="CartoDB dark_matter",
        control_scale=True,
        max_bounds=True,
    )

    fmap.fit_bounds(
        [
            [47.45, -122.46],
            [47.75, -122.20],
        ],
        padding=(20, 20),
    )

    return fmap


def add_neighborhood_boundaries(fmap, boundary_gdf):
    boundary_layer = boundary_gdf.copy()

    # Make sure CRS works for Folium
    if boundary_layer.crs is not None:
        boundary_layer = boundary_layer.to_crs(epsg=4326)
    else:
        boundary_layer = boundary_layer.set_crs(epsg=4326)

    # Use MCPP fields if available, otherwise fall back to old fields
    if "mcpp_neighborhood" in boundary_layer.columns:
        tooltip_fields = ["mcpp_neighborhood"]
        tooltip_aliases = ["MCPP neighborhood:"]
    else:
        tooltip_fields = ["dispatch_neighborhood"]
        tooltip_aliases = ["SPD neighborhood:"]

    if "mcpp_precinct" in boundary_layer.columns:
        tooltip_fields.append("mcpp_precinct")
        tooltip_aliases.append("MCPP precinct:")

    if "unique_call_events" in boundary_layer.columns:
        tooltip_fields.append("unique_call_events")
        tooltip_aliases.append("Unique CAD events:")

    if "dispatch_records" in boundary_layer.columns:
        tooltip_fields.append("dispatch_records")
        tooltip_aliases.append("Dispatch records:")

    # Only include fields that actually exist
    tooltip_fields_final = []
    tooltip_aliases_final = []

    for field, alias in zip(tooltip_fields, tooltip_aliases):
        if field in boundary_layer.columns:
            tooltip_fields_final.append(field)
            tooltip_aliases_final.append(alias)

    folium.GeoJson(
        json.loads(boundary_layer.to_json()),
        name="Official SPD MCPP boundaries",
        style_function=lambda feature: {
            "fillColor": "transparent",
            "color": "#ffffff",
            "weight": 1.0,
            "fillOpacity": 0,
            "opacity": 0.65,
        },
        highlight_function=lambda feature: {
            "fillColor": "#ffd166",
            "color": "#ffd166",
            "weight": 2.5,
            "fillOpacity": 0.15,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=tooltip_fields_final,
            aliases=tooltip_aliases_final,
            sticky=True,
        ),
    ).add_to(fmap)

    return fmap


def make_filtered_event_group_cluster_map(
    map_df,
    title="Filtered SPD calls",
    important_event_groups=IMPORTANT_EVENT_GROUPS,
    include_other_groups=True,
    boundary_gdf=None,
):
    """
    Creates a Folium marker-cluster map for filtered SPD calls.

    Uses official MCPP fields when present:
    - mcpp_neighborhood
    - mcpp_precinct

    Falls back to old dispatch fields when needed.
    """

    if boundary_gdf is None:
        boundary_gdf = mcpp_choropleth_gdf

    plot_df = map_df.copy()

    # Make sure coordinates are numeric
    plot_df[LAT_COL] = pd.to_numeric(plot_df[LAT_COL], errors="coerce")
    plot_df[LON_COL] = pd.to_numeric(plot_df[LON_COL], errors="coerce")

    # Make sure time is datetime
    plot_df[TIME_COLUMN] = pd.to_datetime(plot_df[TIME_COLUMN], errors="coerce")

    plot_df = plot_df.dropna(
        subset=[
            EVENT_ID_COLUMN,
            TIME_COLUMN,
            "event_group",
            LAT_COL,
            LON_COL,
        ]
    ).copy()

    plot_df["event_group"] = (
        plot_df["event_group"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # Prefer official MCPP fields, fall back to old dispatch fields
    if "mcpp_neighborhood" in plot_df.columns:
        neighborhood_col = "mcpp_neighborhood"
    else:
        neighborhood_col = "dispatch_neighborhood"

    if "mcpp_precinct" in plot_df.columns:
        precinct_col = "mcpp_precinct"
    else:
        precinct_col = "dispatch_precinct"

    fmap = make_seattle_folium_base_map()
    fmap = add_neighborhood_boundaries(fmap, boundary_gdf)

    event_groups = (
        plot_df["event_group"]
        .dropna()
        .sort_values()
        .unique()
        .tolist()
    )

    important_set = set(important_event_groups)

    if not include_other_groups:
        event_groups = [
            group for group in event_groups
            if group in important_set
        ]

    for group in event_groups:
        group_df = plot_df[
            plot_df["event_group"] == group
        ].copy()

        show_layer = group in important_set

        feature_group = folium.FeatureGroup(
            name=f"{group} ({len(group_df):,})",
            show=show_layer,
        )

        cluster = MarkerCluster().add_to(feature_group)

        for _, row in group_df.iterrows():
            event_id = row.get(EVENT_ID_COLUMN, "")
            event_time = row.get(TIME_COLUMN, "")
            event_group = row.get("event_group", "")

            mcpp_neighborhood = row.get("mcpp_neighborhood", "")
            mcpp_precinct = row.get("mcpp_precinct", "")

            dispatch_neighborhood = row.get("dispatch_neighborhood", "")
            dispatch_precinct = row.get("dispatch_precinct", "")
            sector = row.get("dispatch_sector", "")
            beat = row.get("dispatch_beat", "")

            priority = row.get("priority", "")
            initial_call_type = row.get("initial_call_type", "")
            final_call_type = row.get("final_call_type", "")

            lat = row.get(LAT_COL)
            lon = row.get(LON_COL)

            display_neighborhood = row.get(neighborhood_col, "")
            display_precinct = row.get(precinct_col, "")

            popup_html = f"""
            <b>CAD Event:</b> {html.escape(str(event_id))}<br>
            <b>Time:</b> {html.escape(str(event_time))}<br>
            <b>Event group:</b> {html.escape(str(event_group))}<br>
            <b>Priority:</b> {html.escape(str(priority))}<br>
            <hr>
            <b>Initial call type:</b> {html.escape(str(initial_call_type))}<br>
            <b>Final call type:</b> {html.escape(str(final_call_type))}<br>
            <hr>
            <b>MCPP neighborhood:</b> {html.escape(str(mcpp_neighborhood))}<br>
            <b>MCPP precinct:</b> {html.escape(str(mcpp_precinct))}<br>
            <hr>
            <b>Original dispatch neighborhood:</b> {html.escape(str(dispatch_neighborhood))}<br>
            <b>Original dispatch precinct:</b> {html.escape(str(dispatch_precinct))}<br>
            <b>Sector:</b> {html.escape(str(sector))}<br>
            <b>Beat:</b> {html.escape(str(beat))}
            """

            folium.Marker(
                location=[lat, lon],
                popup=folium.Popup(popup_html, max_width=400),
                tooltip=f"{event_group} | {display_neighborhood}",
            ).add_to(cluster)

        feature_group.add_to(fmap)

    folium.LayerControl(collapsed=False).add_to(fmap)

    title_html = f"""
    <h3 style="
        position: fixed;
        top: 10px;
        left: 50px;
        z-index: 9999;
        color: white;
        background-color: rgba(0, 0, 0, 0.65);
        padding: 8px 12px;
        border-radius: 4px;
        font-family: Arial;
    ">
        {html.escape(title)}
    </h3>
    """

    fmap.get_root().html.add_child(folium.Element(title_html))

    return fmap

Below we have a map with folium map markers imposed on outlines of the neighborhoods. The accuracy of these points allows us to pinpoint specific areas within Seattle, which is useful for those who not only want a general overview of crime in the city but also a chance to look at more specific parts of town. The ability to hone in on a single recent point, note its proximity to areas you pass through, then examine which neighborhood it resides in gives the user plenty of information while enabling personally relevant analysis. 

In [ ]:
previous_day_events_mcpp = event_mcpp[
    event_mcpp[TIME_COLUMN].dt.normalize() == previous_complete_day
].copy()

previous_day_map = make_filtered_event_group_cluster_map(
    previous_day_events_mcpp,
    title=f"SPD Calls: Previous Complete Day ({previous_complete_day.date()})",
    important_event_groups=IMPORTANT_EVENT_GROUPS,
    include_other_groups=True,
    boundary_gdf=mcpp_choropleth_gdf,
)

previous_day_map

In [ ]:
last_week_start_date = last_week_events[TIME_COLUMN].min().date()
last_week_end_date = last_week_events[TIME_COLUMN].max().date()

last_week_map = make_filtered_event_group_cluster_map(
    last_week_events,
    title=f"SPD Calls: Last 7 Days ({str(last_week_start_date)[5:]} to {str(last_week_end_date)[5:]})",
    important_event_groups=IMPORTANT_EVENT_GROUPS,
    include_other_groups=True,
)

last_week_map

In [ ]:
last_month_important_events = last_month_events[
    last_month_events["event_group"].isin(IMPORTANT_EVENT_GROUPS)
].copy()

last_month_start_date = last_month_important_events[TIME_COLUMN].min().date()
last_month_end_date = last_month_important_events[TIME_COLUMN].max().date()

print(f"Last-month important unique CAD events: {len(last_month_important_events):,}")

last_month_important_map = make_filtered_event_group_cluster_map(
    last_month_important_events,
    title=f"SPD Important Event Groups: Last 30 Days ({str(last_month_start_date)[5:]} to {str(last_month_end_date)[5:]})",
    important_event_groups=IMPORTANT_EVENT_GROUPS,
    include_other_groups=False,
)

last_month_important_map

In [ ]:
folium_window_event_group_summary = (
    last_month_important_events
    .groupby("event_group", as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique")
    )
    .sort_values("unique_call_events", ascending=False)
    .reset_index(drop=True)
)

#folium_window_event_group_summary.head(25)

# Plotly point and density maps

In [ ]:
PLOTLY_SEATTLE_CENTER = {
    "lat": 47.6062,
    "lon": -122.3321,
}

PLOTLY_MAP_STYLE = "carto-darkmatter"

plotly_mappable_events = (
    geo_df[
        geo_df["inside_seattle_bbox"]
        & geo_df[EVENT_ID_COLUMN].notna()
    ]
    .copy()
)

plotly_mappable_events[TIME_COLUMN] = pd.to_datetime(
    plotly_mappable_events[TIME_COLUMN],
    errors="coerce",
)

plotly_mappable_events["event_group"] = (
    plotly_mappable_events["event_group"]
    .astype("string")
    .str.strip()
    .str.lower()
)

plotly_mappable_events["dispatch_neighborhood"] = (
    plotly_mappable_events["dispatch_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

plotly_mappable_events = (
    plotly_mappable_events
    .dropna(subset=[TIME_COLUMN, LAT_COL, LON_COL, "event_group"])
    .sort_values(TIME_COLUMN, ascending=False)
    .drop_duplicates(subset=EVENT_ID_COLUMN)
    .copy()
)

print(f"Mappable unique CAD events: {len(plotly_mappable_events):,}")

plotly_mappable_events.head()

In [ ]:
latest_event_time = plotly_mappable_events[TIME_COLUMN].max()
latest_event_date = latest_event_time.normalize()

previous_complete_day = latest_event_date - pd.Timedelta(days=1)
last_week_start = latest_event_time - pd.Timedelta(days=7)
last_month_start = latest_event_time - pd.Timedelta(days=30)

previous_day_plotly_events = plotly_mappable_events[
    plotly_mappable_events[TIME_COLUMN].dt.normalize() == previous_complete_day
].copy()

last_week_plotly_events = plotly_mappable_events[
    plotly_mappable_events[TIME_COLUMN] >= last_week_start
].copy()

last_month_plotly_events = plotly_mappable_events[
    plotly_mappable_events[TIME_COLUMN] >= last_month_start
].copy()

plotly_window_summary = pd.DataFrame({
    "window": [
        "previous complete day",
        "last 7 days",
        "last 30 days",
    ],
    "unique_call_events": [
        previous_day_plotly_events[EVENT_ID_COLUMN].nunique(),
        last_week_plotly_events[EVENT_ID_COLUMN].nunique(),
        last_month_plotly_events[EVENT_ID_COLUMN].nunique(),
    ],
})

plotly_window_summary

In [ ]:
def geometry_to_line_coordinates(geometry):
    """
    Converts Polygon or MultiPolygon geometry into lon/lat lists
    separated by None values for Plotly line traces.
    """
    lons = []
    lats = []

    if geometry is None or geometry.is_empty:
        return lons, lats

    if geometry.geom_type == "Polygon":
        polygons = [geometry]
    elif geometry.geom_type == "MultiPolygon":
        polygons = list(geometry.geoms)
    else:
        return lons, lats

    for polygon in polygons:
        x, y = polygon.exterior.xy
        lons.extend(list(x))
        lats.extend(list(y))
        lons.append(None)
        lats.append(None)

        for interior in polygon.interiors:
            x, y = interior.xy
            lons.extend(list(x))
            lats.extend(list(y))
            lons.append(None)
            lats.append(None)

    return lons, lats


def add_neighborhood_boundary_lines(fig, boundary_gdf):
    boundary_lons = []
    boundary_lats = []

    for geometry in boundary_gdf.geometry:
        lons, lats = geometry_to_line_coordinates(geometry)
        boundary_lons.extend(lons)
        boundary_lats.extend(lats)

    fig.add_trace(
        go.Scattermapbox(
            lon=boundary_lons,
            lat=boundary_lats,
            mode="lines",
            line=dict(
                width=1,
                color="rgba(255,255,255,0.65)",
            ),
            hoverinfo="skip",
            showlegend=False,
            name="Neighborhood boundaries",
        )
    )

    return fig

def make_plotly_event_group_point_map(
    map_df,
    title,
    important_event_groups=IMPORTANT_EVENT_GROUPS,
    include_all_event_groups=True,
):
    plot_df = map_df.copy()

    if not include_all_event_groups:
        plot_df = plot_df[
            plot_df["event_group"].isin(important_event_groups)
        ].copy()

    important_set = set(important_event_groups)

    ordered_event_groups = (
        important_event_groups
        + sorted(
            group for group in plot_df["event_group"].dropna().unique()
            if group not in important_set
        )
    )

    plot_df["event_group"] = pd.Categorical(
        plot_df["event_group"],
        categories=ordered_event_groups,
        ordered=True,
    )

    plot_df["event_time_display"] = (
        plot_df[TIME_COLUMN]
        .dt.strftime("%Y-%m-%d %H:%M")
    )

    fig = px.scatter_mapbox(
        plot_df,
        lat=LAT_COL,
        lon=LON_COL,
        color="event_group",
        hover_name=EVENT_ID_COLUMN,
        custom_data=[
            "event_time_display",
            "event_group",
            "priority",
            "initial_call_type",
            "final_call_type",
            "dispatch_neighborhood",
            "dispatch_precinct",
            "dispatch_sector",
            "dispatch_beat",
        ],
        center=PLOTLY_SEATTLE_CENTER,
        zoom=10,
        mapbox_style=PLOTLY_MAP_STYLE,
        title=title,
        opacity=0.70,
        category_orders={
            "event_group": ordered_event_groups,
        },
    )

    fig.update_traces(
        marker=dict(size=7),
        hovertemplate=(
            "<b>CAD Event:</b> %{hovertext}<br>"
            "<b>Time:</b> %{customdata[0]}<br>"
            "<b>Event group:</b> %{customdata[1]}<br>"
            "<b>Priority:</b> %{customdata[2]}<br>"
            "<br>"
            "<b>Initial call type:</b> %{customdata[3]}<br>"
            "<b>Final call type:</b> %{customdata[4]}<br>"
            "<br>"
            "<b>Neighborhood:</b> %{customdata[5]}<br>"
            "<b>Precinct:</b> %{customdata[6]}<br>"
            "<b>Sector:</b> %{customdata[7]}<br>"
            "<b>Beat:</b> %{customdata[8]}"
            "<extra></extra>"
        ),
    )

    # Hide non-important event groups by default, but keep them clickable.
    for trace in fig.data:
        if trace.name not in important_set:
            trace.visible = "legendonly"

    fig = add_neighborhood_boundary_lines(
        fig,
        mcpp_choropleth_gdf,
    )

    fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        legend_title_text="Event group",
    )

    return fig

Below we have a point map of the latest complete day of data. The advantage of the plotly version of this map is the coloring of the points, instantly you see which types of crimes are occuring in specific areas. Right away you can see the prevalence of theft in the downtown area, sparse distribution of assaults throughout the city, and the amount of narcotics calls that surround the downtown area. 

In [ ]:
fig = make_plotly_event_group_point_map(
    previous_day_plotly_events,
    title=f"SPD Calls by Event Group: Previous Complete Day ({previous_complete_day.date()})",
    important_event_groups=IMPORTANT_EVENT_GROUPS,
    include_all_event_groups=True,
)

fig.show()

Below we can see the same figure as above but for a larger timeframe, while there is a bit of visual clutter it is also clear what types of crimes occur in each region of the map.

In [ ]:
fig = make_plotly_event_group_point_map(
    last_week_plotly_events,
    title="SPD Calls by Event Group: Last 7 Days",
    important_event_groups=IMPORTANT_EVENT_GROUPS,
    include_all_event_groups=True,
)

fig.show()

Below we expand the timeframe of the figure to 30 days, increasing the amount of time we can analyze at the cost of a little more clutter. Some things that stick out are the gridded visual pattern of narcotics calls, implying that the presence of narcotics usually occurs on specific streets. Assualts share some of the same street locked pattern as narcotics but to a lesser degree, and no other crimes demonstrate that pattern.

In [ ]:
last_month_important_plotly_events = last_month_plotly_events[
    last_month_plotly_events["event_group"].isin(IMPORTANT_EVENT_GROUPS)
].copy()

print(f"Last-month important unique CAD events: {len(last_month_important_plotly_events):,}")

fig = make_plotly_event_group_point_map(
    last_month_important_plotly_events,
    title="Important SPD Event Groups: Last 30 Days",
    important_event_groups=IMPORTANT_EVENT_GROUPS,
    include_all_event_groups=False,
)

fig.show()

Below we can see a plotly density map of the city's calls, notice how this map omits the types of crimes and becomes difficult to interpret when zoomed out.

In [ ]:
density_important_events = last_month_plotly_events[
    last_month_plotly_events["event_group"].isin(IMPORTANT_EVENT_GROUPS)
].copy()

fig = px.density_mapbox(
    density_important_events,
    lat=LAT_COL,
    lon=LON_COL,
    radius=10,
    center=PLOTLY_SEATTLE_CENTER,
    zoom=10,
    mapbox_style=PLOTLY_MAP_STYLE,
    title="Density of Important SPD Event Groups: Last 30 Days",
)

fig = add_neighborhood_boundary_lines(
    fig,
    mcpp_choropleth_gdf,
)

fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
)

fig.show()

In [ ]:
choropleth_events = geo_df.copy()

choropleth_events[TIME_COLUMN] = pd.to_datetime(
    choropleth_events[TIME_COLUMN],
    errors="coerce",
)

choropleth_events["event_group"] = (
    choropleth_events["event_group"]
    .astype("string")
    .str.strip()
    .str.lower()
)

choropleth_events["dispatch_neighborhood"] = (
    choropleth_events["dispatch_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

choropleth_events = (
    choropleth_events
    .dropna(subset=[EVENT_ID_COLUMN, TIME_COLUMN, "dispatch_neighborhood"])
    .query("dispatch_neighborhood != '-'")
    .query("dispatch_neighborhood != ''")
    .sort_values(TIME_COLUMN, ascending=False)
    .drop_duplicates(subset=EVENT_ID_COLUMN)
    .copy()
)

print(f"Unique CAD events available for choropleths: {len(choropleth_events):,}")

choropleth_events.head()

In [ ]:
def get_neighborhood_column(df):
    """
    Prefer official MCPP geography when available.
    Fall back to old dispatch_neighborhood only if needed.
    """
    if "mcpp_neighborhood" in df.columns:
        return "mcpp_neighborhood"
    if "dispatch_neighborhood" in df.columns:
        return "dispatch_neighborhood"

    raise KeyError(
        "No neighborhood column found. Expected 'mcpp_neighborhood' or "
        "'dispatch_neighborhood'. "
        f"Available columns: {df.columns.tolist()}"
    )


def get_precinct_column(df):
    """
    Prefer official MCPP precinct when available.
    Fall back to old dispatch_precinct only if needed.
    """
    if "mcpp_precinct" in df.columns:
        return "mcpp_precinct"
    if "dispatch_precinct" in df.columns:
        return "dispatch_precinct"

    return None


def count_events_by_neighborhood(
    events_df,
    count_col_name="event_count",
    neighborhood_col=None,
):
    """
    Counts unique CAD events by neighborhood.

    Uses mcpp_neighborhood when available.
    Falls back to dispatch_neighborhood for older dataframes.
    """

    if neighborhood_col is None:
        neighborhood_col = get_neighborhood_column(events_df)

    events_clean = events_df.copy()

    events_clean[neighborhood_col] = (
        events_clean[neighborhood_col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    events_clean = events_clean[
        events_clean[neighborhood_col].notna()
        & ~events_clean[neighborhood_col].isin(["-", "", "nan", "unknown"])
    ].copy()

    counts = (
        events_clean
        .groupby(neighborhood_col, as_index=False)
        .agg(
            **{
                count_col_name: (EVENT_ID_COLUMN, "nunique")
            }
        )
    )

    return counts


def build_neighborhood_choropleth_data(
    boundary_gdf,
    events_df,
    count_col_name="event_count",
    neighborhood_col=None,
):
    """
    Builds choropleth-ready GeoDataFrame.

    For the new workflow:
    - boundary_gdf should usually be mcpp_boundaries or mcpp_choropleth_gdf
    - events_df should usually be event_mcpp
    - merge key should be mcpp_neighborhood
    """

    boundary_base = boundary_gdf.copy()

    if neighborhood_col is None:
        neighborhood_col = get_neighborhood_column(boundary_base)

    # Make sure events are counted using the same neighborhood field
    counts = count_events_by_neighborhood(
        events_df,
        count_col_name=count_col_name,
        neighborhood_col=neighborhood_col,
    )

    boundary_base[neighborhood_col] = (
        boundary_base[neighborhood_col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    counts[neighborhood_col] = (
        counts[neighborhood_col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # Prevent column collisions if the boundary file already has this metric
    if count_col_name in boundary_base.columns:
        boundary_base = boundary_base.drop(columns=[count_col_name])

    out = boundary_base.merge(
        counts,
        on=neighborhood_col,
        how="left",
    )

    out[count_col_name] = (
        out[count_col_name]
        .fillna(0)
        .astype(int)
    )

    return out


def make_neighborhood_choropleth(
    choropleth_gdf,
    color_col,
    title,
    color_label,
    color_scale="Viridis",
    value_format=":,.0f",
):
    """
    Makes a Plotly choropleth using official MCPP geography when available.

    Uses:
    - mcpp_neighborhood / mcpp_precinct for new MCPP workflow
    - dispatch_neighborhood / dispatch_precinct as fallback
    """

    plot_gdf = choropleth_gdf.copy()

    neighborhood_col = get_neighborhood_column(plot_gdf)
    precinct_col = get_precinct_column(plot_gdf)

    plot_gdf[neighborhood_col] = (
        plot_gdf[neighborhood_col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # Use objectid when available because it is safer than names for GeoJSON matching
    if "objectid" in plot_gdf.columns:
        plot_gdf["plot_feature_id"] = plot_gdf["objectid"].astype(str)
        feature_id_key = "properties.plot_feature_id"
    else:
        plot_gdf["plot_feature_id"] = plot_gdf[neighborhood_col].astype(str)
        feature_id_key = "properties.plot_feature_id"

    geojson = json.loads(plot_gdf.to_json())

    custom_data_cols = [
        neighborhood_col,
        color_col,
    ]

    if precinct_col is not None:
        custom_data_cols.append(precinct_col)

    if "dispatch_records" in plot_gdf.columns:
        custom_data_cols.append("dispatch_records")

    fig = px.choropleth_mapbox(
        plot_gdf,
        geojson=geojson,
        locations="plot_feature_id",
        featureidkey=feature_id_key,
        color=color_col,
        hover_name=neighborhood_col,
        custom_data=custom_data_cols,
        center=PLOTLY_SEATTLE_CENTER,
        zoom=10,
        mapbox_style=PLOTLY_MAP_STYLE,
        color_continuous_scale=color_scale,
        title=title,
        opacity=0.75,
    )

    if precinct_col is not None and "dispatch_records" in plot_gdf.columns:
        fig.update_traces(
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                f"{color_label}: " + f"%{{customdata[1]{value_format}}}<br>"
                "Precinct: %{customdata[2]}<br>"
                "Dispatch records: %{customdata[3]:,}"
                "<extra></extra>"
            )
        )

    elif precinct_col is not None:
        fig.update_traces(
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                f"{color_label}: " + f"%{{customdata[1]{value_format}}}<br>"
                "Precinct: %{customdata[2]}"
                "<extra></extra>"
            )
        )

    elif "dispatch_records" in plot_gdf.columns:
        fig.update_traces(
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                f"{color_label}: " + f"%{{customdata[1]{value_format}}}<br>"
                "Dispatch records: %{customdata[2]:,}"
                "<extra></extra>"
            )
        )

    else:
        fig.update_traces(
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                f"{color_label}: " + f"%{{customdata[1]{value_format}}}"
                "<extra></extra>"
            )
        )

    fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        coloraxis_colorbar_title=color_label,
    )

    return fig

Below we can see an alternate color scheme for a chlorolpleth map of the volume of calls in each neighborhood

In [ ]:
total_neighborhood_choropleth = build_neighborhood_choropleth_data(
    boundary_gdf=mcpp_choropleth_gdf,
    events_df=event_mcpp,
    count_col_name="total_event_count",
    neighborhood_col="mcpp_neighborhood",
)

fig = make_neighborhood_choropleth(
    choropleth_gdf=total_neighborhood_choropleth,
    color_col="total_event_count",
    title="SPD Unique CAD Events by Official MCPP Neighborhood",
    color_label="Unique CAD events",
)

fig.show()

Below we can see the same figure as above but for violent or drug related crimes

In [ ]:
important_choropleth_events = event_mcpp[
    event_mcpp["event_group"].isin(IMPORTANT_EVENT_GROUPS)
].copy()

important_neighborhood_choropleth = build_neighborhood_choropleth_data(
    boundary_gdf=mcpp_choropleth_gdf,
    events_df=important_choropleth_events,
    count_col_name="important_event_count",
    neighborhood_col="mcpp_neighborhood",
)

fig = make_neighborhood_choropleth(
    choropleth_gdf=important_neighborhood_choropleth,
    color_col="important_event_count",
    title="Important SPD Event Groups by Official MCPP Neighborhood",
    color_label="Important event count",
    color_scale="Viridis",
)

fig.show()

Below we can see the share of violent or drug related calls in the city on a chloropleth map.

In [ ]:
total_counts = count_events_by_neighborhood(
    event_mcpp,
    count_col_name="total_event_count",
    neighborhood_col="mcpp_neighborhood",
)

important_choropleth_events = event_mcpp[
    event_mcpp["event_group"].isin(IMPORTANT_EVENT_GROUPS)
].copy()

important_counts = count_events_by_neighborhood(
    important_choropleth_events,
    count_col_name="important_event_count",
    neighborhood_col="mcpp_neighborhood",
)

important_share_counts = total_counts.merge(
    important_counts,
    on="mcpp_neighborhood",
    how="left",
)

important_share_counts["important_event_count"] = (
    important_share_counts["important_event_count"]
    .fillna(0)
    .astype(int)
)

important_share_counts["important_event_share"] = (
    important_share_counts["important_event_count"]
    / important_share_counts["total_event_count"]
    * 100
)

important_share_choropleth = mcpp_choropleth_gdf.merge(
    important_share_counts,
    on="mcpp_neighborhood",
    how="left",
)

important_share_choropleth["total_event_count"] = (
    important_share_choropleth["total_event_count"]
    .fillna(0)
    .astype(int)
)

important_share_choropleth["important_event_count"] = (
    important_share_choropleth["important_event_count"]
    .fillna(0)
    .astype(int)
)

important_share_choropleth["important_event_share"] = (
    important_share_choropleth["important_event_share"]
    .fillna(0)
)

fig = make_neighborhood_choropleth(
    important_share_choropleth,
    color_col="important_event_share",
    title="Share of SPD Calls in Important Event Groups by Official MCPP Neighborhood",
    color_label="Important event share (%)",
    color_scale="Viridis",
    value_format=":.1f",
)

fig.show()

Below we see a chloropleth map for the number of unique events that have occured within the last 7 days.

In [ ]:
latest_event_time = event_mcpp[TIME_COLUMN].max()

last_week_start = latest_event_time - pd.Timedelta(days=7)
last_month_start = latest_event_time - pd.Timedelta(days=30)

last_week_choropleth_events = event_mcpp[
    event_mcpp[TIME_COLUMN] >= last_week_start
].copy()

last_month_choropleth_events = event_mcpp[
    event_mcpp[TIME_COLUMN] >= last_month_start
].copy()

recent_window_summary = pd.DataFrame({
    "window": [
        "last 7 days",
        "last 30 days",
    ],
    "unique_call_events": [
        last_week_choropleth_events[EVENT_ID_COLUMN].nunique(),
        last_month_choropleth_events[EVENT_ID_COLUMN].nunique(),
    ],
})

display(recent_window_summary)


last_week_neighborhood_choropleth = build_neighborhood_choropleth_data(
    boundary_gdf=mcpp_choropleth_gdf,
    events_df=last_week_choropleth_events,
    count_col_name="last_week_event_count",
    neighborhood_col="mcpp_neighborhood",
)

fig = make_neighborhood_choropleth(
    choropleth_gdf=last_week_neighborhood_choropleth,
    color_col="last_week_event_count",
    title="SPD Unique CAD Events by Official MCPP Neighborhood: Last 7 Days",
    color_label="Last 7 days events",
)

fig.show()

In [ ]:
last_month_neighborhood_choropleth = build_neighborhood_choropleth_data(
    boundary_gdf=mcpp_choropleth_gdf,
    events_df=last_month_choropleth_events,
    count_col_name="last_month_event_count",
    neighborhood_col="mcpp_neighborhood",
)

fig = make_neighborhood_choropleth(
    choropleth_gdf=last_month_neighborhood_choropleth,
    color_col="last_month_event_count",
    title="SPD Unique CAD Events by Official MCPP Neighborhood: Last 30 Days",
    color_label="Last 30 days events",
)

fig.show()

Below we can explore the number of calls within each neighborhood in specific categories. By default the narcotics calls are shown in a chloropleth map, in which we can see the very high relative number of narcotics crimes occuring in the area surrounding downtown.

In [ ]:
TARGET_EVENT_GROUP = "narcotics"

target_event_group_events = event_mcpp[
    event_mcpp["event_group"] == TARGET_EVENT_GROUP
].copy()

target_event_group_choropleth = build_neighborhood_choropleth_data(
    boundary_gdf=mcpp_choropleth_gdf,
    events_df=target_event_group_events,
    count_col_name="target_event_count",
    neighborhood_col="mcpp_neighborhood",
)

fig = make_neighborhood_choropleth(
    choropleth_gdf=target_event_group_choropleth,
    color_col="target_event_count",
    title=f"SPD {TARGET_EVENT_GROUP.title()} Calls by Official MCPP Neighborhood",
    color_label=f"{TARGET_EVENT_GROUP.title()} events",
)

fig.show()

The map below combines the chloropleth mapping of share of events that are violent/drug related calls, and includes points for each event occuring within the last available day. In the map we can see which neighborhoods have a high proportion of important events (i.e. a high proportion of the calls within the neighborhood were in important event groups) and which types of crimes have recently been committed within each neighborhood and where.

In [ ]:
previous_complete_day = latest_event_time.normalize() - pd.Timedelta(days=1)

previous_day_points = event_mcpp[
    event_mcpp[TIME_COLUMN].dt.normalize() == previous_complete_day
].copy()

previous_day_important_points = previous_day_points[
    previous_day_points["event_group"].isin(IMPORTANT_EVENT_GROUPS)
].copy()

previous_day_important_points["event_time_display"] = (
    previous_day_important_points[TIME_COLUMN]
    .dt.strftime("%Y-%m-%d %H:%M")
)

print(f"Previous-day important points: {len(previous_day_important_points):,}")

# Make sure optional hover columns exist
for col in [
    "priority",
    "initial_call_type",
    "final_call_type",
    "dispatch_neighborhood",
    "dispatch_precinct",
    "dispatch_sector",
    "dispatch_beat",
]:
    if col not in previous_day_important_points.columns:
        previous_day_important_points[col] = pd.NA

base_gdf = important_share_choropleth.copy()

base_gdf["plot_feature_id"] = base_gdf["objectid"].astype(str)

base_geojson = json.loads(base_gdf.to_json())

fig = px.choropleth_mapbox(
    base_gdf,
    geojson=base_geojson,
    locations="plot_feature_id",
    featureidkey="properties.plot_feature_id",
    color="important_event_share",
    hover_name="mcpp_neighborhood",
    custom_data=[
        "mcpp_neighborhood",
        "mcpp_precinct",
        "important_event_share",
        "important_event_count",
        "total_event_count",
    ],
    center=PLOTLY_SEATTLE_CENTER,
    zoom=10,
    mapbox_style=PLOTLY_MAP_STYLE,
    color_continuous_scale="Viridis",
    title="Important Event Share by MCPP Neighborhood with Previous-Day Important Calls",
    opacity=0.65,
)

fig.update_traces(
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "MCPP precinct: %{customdata[1]}<br>"
        "Important event share: %{customdata[2]:.1f}%<br>"
        "Important events: %{customdata[3]:,}<br>"
        "Total events: %{customdata[4]:,}"
        "<extra></extra>"
    )
)

for group in IMPORTANT_EVENT_GROUPS:
    group_points = previous_day_important_points[
        previous_day_important_points["event_group"] == group
    ].copy()

    if group_points.empty:
        continue

    fig.add_trace(
        go.Scattermapbox(
            lat=group_points[LAT_COL],
            lon=group_points[LON_COL],
            mode="markers",
            name=group,
            marker=dict(
                size=8,
                opacity=0.85,
            ),
            customdata=group_points[
                [
                    EVENT_ID_COLUMN,
                    "event_time_display",
                    "event_group",
                    "priority",
                    "initial_call_type",
                    "final_call_type",
                    "mcpp_neighborhood",
                    "mcpp_precinct",
                    "dispatch_neighborhood",
                    "dispatch_precinct",
                    "dispatch_sector",
                    "dispatch_beat",
                ]
            ],
            hovertemplate=(
                "<b>CAD Event:</b> %{customdata[0]}<br>"
                "<b>Time:</b> %{customdata[1]}<br>"
                "<b>Event group:</b> %{customdata[2]}<br>"
                "<b>Priority:</b> %{customdata[3]}<br>"
                "<br>"
                "<b>Initial call type:</b> %{customdata[4]}<br>"
                "<b>Final call type:</b> %{customdata[5]}<br>"
                "<br>"
                "<b>MCPP neighborhood:</b> %{customdata[6]}<br>"
                "<b>MCPP precinct:</b> %{customdata[7]}<br>"
                "<br>"
                "<b>Original dispatch neighborhood:</b> %{customdata[8]}<br>"
                "<b>Original dispatch precinct:</b> %{customdata[9]}<br>"
                "<b>Sector:</b> %{customdata[10]}<br>"
                "<b>Beat:</b> %{customdata[11]}"
                "<extra></extra>"
            ),
        )
    )

fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    legend_title_text="Previous-day important calls",
    coloraxis_colorbar_title="Important event share (%)",
)

fig.show()